In [1]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

load_dotenv()

PROVIDER = "groq"

LLM_CONFIGS = {
    "groq": {
        "api_key": os.getenv("GROQ_API_KEY"),
        "base_url": "https://api.groq.com/openai/v1",
        "model": "openai/gpt-oss-120b",
    },
    "openai": {
        "api_key": os.getenv("OPENAI_API_KEY"),
        "base_url": None,
        "model": "gpt-4o-mini",
    },
}

def get_llm(temperature: float = 0.5, seed: int | None = None) -> ChatOpenAI:
    config = LLM_CONFIGS[PROVIDER]
    if not config["api_key"]:
        raise RuntimeError(f".env 파일에 {PROVIDER.upper()}_API_KEY 가 없습니다.")
    return ChatOpenAI(**config, temperature=temperature, seed=seed, max_retries=5)

llm = get_llm(temperature=0)
print(f"{PROVIDER} / {llm.model_name}")

groq / openai/gpt-oss-120b


## 2-1

In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()

prompt = PromptTemplate(
    template="한국의 {interest} 분야에서 유명한 장소나 활동 5가지를 알려주세요.\n{format_instructions}",
    input_variables=["interest"],
    partial_variables={"format_instructions": format_instructions},
)

chain = prompt | llm | output_parser

interest = input("관심 분야를 입력하세요 (예: 음식, 스포츠, 영화): ")
result = chain.invoke({"interest": interest})

print(f"[{interest}] 관련 추천 5가지")
print(result)  
for i, item in enumerate(result, 1):
    print(f"{i}. {item}")

[음식] 관련 추천 5가지
['광장시장', '남대문시장', '전주한옥마을 비빔밥 체험', '김치 만들기 워크숍', '이태원 불고기 거리']
1. 광장시장
2. 남대문시장
3. 전주한옥마을 비빔밥 체험
4. 김치 만들기 워크숍
5. 이태원 불고기 거리


## 2-2

In [ ]:
from enum import Enum
from langchain_classic.output_parsers import EnumOutputParser, OutputFixingParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException


class Sentiment(Enum):
    """영화 리뷰 감정 분류 값"""
    POSITIVE = "긍정"
    NEGATIVE = "부정"
    NEUTRAL = "보통"


parser = EnumOutputParser(enum=Sentiment)
format_instructions = parser.get_format_instructions()
print(format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 영화 리뷰 감정 분류 전문가입니다. 리뷰를 읽고 감정을 분류하세요. 설명 없이 값만 답하세요.\n{format_instructions}"),
    ("user", "리뷰: {review}"),
]).partial(format_instructions=format_instructions)


fixing_parser = OutputFixingParser.from_llm(llm=llm, parser=parser, max_retries=1)
chain = prompt | llm | fixing_parser

reviews = [
    "이 영화 정말 재미없었어요. 시간 낭비였습니다.",
    "배우들의 연기가 훌륭하고 스토리도 감동적이었어요!",
    "그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.",
]

for review in reviews:
    print(f"리뷰: {review}")
    try:
        result = chain.invoke({"review": review})
        print(f"  분류 결과: {result.value}")
    except OutputParserException as e:
        print(f"  파싱 오류: {str(e).splitlines()[0]}")
    print("-" * 50)

Select one of the following options: 긍정, 부정, 보통
리뷰: 이 영화 정말 재미없었어요. 시간 낭비였습니다.
  분류 결과: 부정
--------------------------------------------------
리뷰: 배우들의 연기가 훌륭하고 스토리도 감동적이었어요!
  분류 결과: 긍정
--------------------------------------------------
리뷰: 그냥 무난한 영화였습니다. 나쁘지도 좋지도 않아요.
  분류 결과: 보통
--------------------------------------------------


## 2-3

In [4]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


class StudentInfo(BaseModel):
    """자기소개에서 추출한 학생 정보"""
    name: str = Field(..., description="학생의 이름")
    age: int = Field(..., description="학생의 나이 (숫자만)")
    major: str = Field(..., description="학생의 전공")
    hobbies: list[str] = Field(..., description="학생의 취미 목록")
    goal: str = Field(..., description="학생의 앞으로의 목표")


parser = PydanticOutputParser(pydantic_object=StudentInfo)
format_instructions = parser.get_format_instructions()

prompt = ChatPromptTemplate.from_messages([
    ("system", "자기소개 글에서 정보를 추출하는 AI입니다. 아래 형식을 정확히 따르세요.\n{format_instructions}"),
    ("user", "{text}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | parser

text = ("안녕하세요! 저는 김민수이고 22살입니다. 컴퓨터공학을 전공하고 있어요. "
        "취미로는 게임하기, 영화보기, 코딩을 좋아합니다. 앞으로 훌륭한 개발자가 되는 것이 목표입니다.")

student = chain.invoke({"text": text})
print(type(student))
print(student)
print(f"이름: {student.name} / 나이: {student.age} / 전공: {student.major}")
print(f"취미: {student.hobbies}")
print(f"목표: {student.goal}")

<class '__main__.StudentInfo'>
name='김민수' age=22 major='컴퓨터공학' hobbies=['게임하기', '영화보기', '코딩'] goal='훌륭한 개발자가 되는 것'
이름: 김민수 / 나이: 22 / 전공: 컴퓨터공학
취미: ['게임하기', '영화보기', '코딩']
목표: 훌륭한 개발자가 되는 것


## 2-4

In [5]:
import json
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.exceptions import OutputParserException

response_schemas = [
    ResponseSchema(name="destination", description="여행지"),
    ResponseSchema(name="duration", description="여행 기간 (예: 2박 3일)"),
    ResponseSchema(name="budget", description="총 사용 금액 (예: 30만원)"),
    ResponseSchema(name="rating", description="5점 만점 기준 만족도 점수 (숫자만)"),
    ResponseSchema(name="activities", description="여행에서 한 주요 활동 목록 (리스트)"),
]

parser = StructuredOutputParser.from_response_schemas(response_schemas)
format_instructions = parser.get_format_instructions()
print(format_instructions)

prompt = ChatPromptTemplate.from_messages([
    ("system", "여행 후기에서 정보를 추출하는 AI입니다. 아래 형식을 정확히 따르세요.\n{format_instructions}"),
    ("user", "{text}"),
]).partial(format_instructions=format_instructions)

chain = prompt | llm | parser

text = ("지난 주에 부산으로 2박 3일 여행을 다녀왔어요. 총 30만원 정도 썼는데 "
        "해운대에서 바다구경하고, 자갈치시장에서 회 먹고, 감천문화마을도 구경했어요. "
        "정말 만족스러운 여행이었습니다. 5점 만점에 4점 정도 줄 수 있을 것 같아요.")

try:
    result = chain.invoke({"text": text})
    print(type(result))  # dict
    print(json.dumps(result, indent=4, ensure_ascii=False))
    print("여행지:", result["destination"])
    print("활동:", result["activities"])
except OutputParserException as e:
    print(f"파싱 오류: {str(e).splitlines()[0]}")

The output should be a markdown code snippet formatted in the following schema, including the leading and trailing "```json" and "```":

```json
{
	"destination": string  // 여행지
	"duration": string  // 여행 기간 (예: 2박 3일)
	"budget": string  // 총 사용 금액 (예: 30만원)
	"rating": string  // 5점 만점 기준 만족도 점수 (숫자만)
	"activities": string  // 여행에서 한 주요 활동 목록 (리스트)
}
```
<class 'dict'>
{
    "destination": "부산",
    "duration": "2박 3일",
    "budget": "30만원",
    "rating": "4",
    "activities": "해운대에서 바다구경, 자갈치시장에서 회 먹기, 감천문화마을 구경"
}
여행지: 부산
활동: 해운대에서 바다구경, 자갈치시장에서 회 먹기, 감천문화마을 구경
